# 📘 Module 5.3 — Fine-Tuning & Prompt Engineering

**Goal:** Learn how to adapt pretrained LLMs for ADAS-specific tasks.

## Two Ways to Use LLMs

| Approach | Data Needed | Compute | When to Use |
|----------|------------|---------|-------------|
| **Prompt Engineering** | None | Low | Quick experiments, prototyping |
| **Fine-Tuning** | Hundreds+ examples | Medium-High | Domain-specific performance |
| **LoRA/QLoRA** | Hundreds+ examples | Low-Medium | Efficient fine-tuning |

---

In [1]:
import torch
import torch.nn as nn
import numpy as np

## 1. Prompt Engineering for ADAS

Prompt engineering = crafting the right input text to get useful outputs from an LLM **without changing its weights**.

In [2]:
# --- ADAS Prompt Templates ---

prompts = {
    "Scene Description": """
You are an ADAS perception system. Describe the driving scene:
- Objects detected: {objects}
- Weather: {weather}
- Road type: {road_type}

Provide a safety assessment and recommended actions.
""",
    
    "Incident Analysis": """
Analyze the following driving incident:
- Event: {event}
- Speed: {speed} km/h
- Time: {time}
- Sensor data: {sensor_data}

Determine: 1) Root cause, 2) ADAS response quality, 3) Improvement suggestions.
""",
    
    "Few-Shot Classification": """
Classify the driving scenario into one of: SAFE, CAUTION, DANGER.

Examples:
- Clear weather, empty highway, 100km/h → SAFE
- Rain, school zone, children present → DANGER
- Light traffic, urban road, green light → SAFE
- Fog, highway merge, heavy traffic → CAUTION

Now classify:
- {scenario} →
"""
}

# Fill template
scene_prompt = prompts["Scene Description"].format(
    objects="2 cars, 1 pedestrian, 1 cyclist",
    weather="rainy",
    road_type="urban intersection"
)
print(scene_prompt)
print("---")
print("💡 Send this to GPT-4/Claude/LLaMA for a detailed response")


You are an ADAS perception system. Describe the driving scene:
- Objects detected: 2 cars, 1 pedestrian, 1 cyclist
- Weather: rainy
- Road type: urban intersection

Provide a safety assessment and recommended actions.

---
💡 Send this to GPT-4/Claude/LLaMA for a detailed response


## 2. LoRA: Low-Rank Adaptation

**LoRA** (Hu et al., 2021) is the most popular efficient fine-tuning method:
- Freeze the original model weights
- Add small trainable "adapter" matrices
- Reduces trainable parameters by **99%+**

$$W' = W + \Delta W = W + BA$$

Where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, and $r \ll d$ (low rank).

In [3]:
class LoRALinear(nn.Module):
    """Linear layer with LoRA adaptation — from scratch implementation."""
    
    def __init__(self, original_linear, rank=8, alpha=16):
        super().__init__()
        self.original = original_linear
        self.original.weight.requires_grad = False  # Freeze original!
        
        in_features = original_linear.in_features
        out_features = original_linear.out_features
        
        # LoRA matrices (small and trainable)
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.scale = alpha / rank
    
    def forward(self, x):
        # Original output + LoRA delta
        original_out = self.original(x)
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scale
        return original_out + lora_out

# Example: Apply LoRA to a large linear layer
original = nn.Linear(4096, 4096)  # ~16.7M parameters
lora_layer = LoRALinear(original, rank=8)

original_params = sum(p.numel() for p in original.parameters())
lora_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)

print(f"Original parameters: {original_params:,}")
print(f"LoRA trainable parameters: {lora_params:,}")
print(f"Reduction: {(1 - lora_params/original_params)*100:.1f}%")
print(f"\n→ We only train {lora_params:,} params instead of {original_params:,}!")

Original parameters: 16,781,312
LoRA trainable parameters: 69,632
Reduction: 99.6%

→ We only train 69,632 params instead of 16,781,312!


## 3. Fine-Tuning Pipeline (Conceptual)

```python
# Pseudocode for fine-tuning with Hugging Face
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# 1. Load pretrained model
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3-8b")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3-8b")

# 2. Apply LoRA
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"])
model = get_peft_model(model, lora_config)

# 3. Prepare ADAS-specific dataset
dataset = load_dataset("your_adas_driving_scenarios")

# 4. Train
trainer = Trainer(model=model, train_dataset=dataset, args=training_args)
trainer.train()

# 5. Save adapted weights (only LoRA weights, very small!)
model.save_pretrained("adas-llama-lora")
```

In [4]:
# --- Comparison of LLMs for ADAS ---
print("LLMs Relevant to ADAS:")
print("=" * 60)
models_info = [
    ("GPT-4o", "Closed", "Best general performance"),
    ("Claude 3.5", "Closed", "Strong reasoning"),
    ("LLaMA-3 8B", "Open", "Good for fine-tuning"),
    ("LLaMA-3 70B", "Open", "High quality, needs GPUs"),
    ("Mistral 7B", "Open", "Efficient, good quality"),
    ("Phi-3 mini", "Open", "Edge deployment ready"),
    ("Qwen-2", "Open", "Strong multilingual"),
]

for name, access, note in models_info:
    print(f"  {name:<15} [{access:<6}] {note}")

LLMs Relevant to ADAS:
  GPT-4o          [Closed] Best general performance
  Claude 3.5      [Closed] Strong reasoning
  LLaMA-3 8B      [Open  ] Good for fine-tuning
  LLaMA-3 70B     [Open  ] High quality, needs GPUs
  Mistral 7B      [Open  ] Efficient, good quality
  Phi-3 mini      [Open  ] Edge deployment ready
  Qwen-2          [Open  ] Strong multilingual


---
## ✅ Key Takeaways

1. **Prompt engineering** is the fastest way to use LLMs — no training needed
2. **LoRA** enables efficient fine-tuning by training only ~0.1% of parameters
3. **Fine-tuning** adapts a general LLM to your specific domain (ADAS scenarios)
4. **Open-source LLMs** (LLaMA, Mistral) can be deployed on-premise for ADAS
5. For edge deployment, consider smaller models (Phi-3, Mistral 7B)

---
## 📖 Next Steps
➡️ **Next module:** [06_VLMs/01_clip_multimodal_basics.ipynb](../06_VLMs/01_clip_multimodal_basics.ipynb) — Vision-Language Models (CLIP)